# MLB Moneyline Model Lab — Missingness, Statcast, Batted-Ball Features

This notebook is a refreshed development/QA notebook for the TetheredAI MLB moneyline pipeline.

It focuses on:

1. Validating the feature file produced by the GCP pipeline.
2. Dropping very-high-missing features, especially unstable ratio features such as stolen-base success rate.
3. Keeping moderate-missing Statcast starter features with imputation and missingness indicators.
4. Detecting and using team batted-ball quality features such as average exit velocity and average batted-ball distance when available.
5. Comparing model families by log loss, Brier score, AUC, accuracy, and calibration.
6. Exporting a champion model bundle only when you explicitly approve it.

Important: this notebook trains only on rows where `target_home_win` is known. Upcoming/unfinal games are excluded from training and can be scored later by the production scoring job.

## Modeling philosophy

For betting, probability quality matters more than raw accuracy. The key metrics are:

- **Log loss**: punishes confident wrong probabilities.
- **Brier score**: squared probability error.
- **Calibration**: whether games predicted at 55% actually win near 55% of the time.
- **AUC**: ranking/discrimination ability.

Accuracy is still shown, but do not choose a champion model based only on accuracy.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import re
import sqlite3
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# Project root detection works whether notebook is launched from project root or /notebooks.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "mlb_game_features.parquet"
DB_PATH = PROJECT_ROOT / "data" / "odds.db"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURES_PATH:", FEATURES_PATH)
print("DB_PATH:", DB_PATH)
print("MODEL_DIR:", MODEL_DIR)

In [ ]:
# Main modeling controls
TARGET_COL = "target_home_win"
DATE_COL_CANDIDATES = ["game_datetime_utc", "official_date", "game_date"]

TEST_SIZE = 0.20
RANDOM_STATE = 42

# Missingness policy
DROP_IF_MISSING_GT = 0.35
ADD_MISSING_FLAG_IF_MISSING_GT = 0.02

# Market features are useful later, but default to False so we can test pure baseball signal first.
USE_MARKET_FEATURES = False

# Runtime controls
RUN_XGBOOST = True
RUN_LIGHTGBM = True
RUN_SVM = False  # set True later; SVM can be slow.
USE_RANDOMIZED_SEARCH = True
N_ITER = 20
CV_SPLITS = 3

# Export control. Leave False until you review results.
APPROVE_EXPORT = False
CHAMPION_KEY = None  # None = use best log_loss row. Or set tuple like ("enhanced_no_market", "xgboost").
CHAMPION_EXPORT_NAME = "mlb_moneyline_champion"

## Load feature file

In [ ]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f"Feature file not found: {FEATURES_PATH}")

features = pd.read_parquet(FEATURES_PATH)
print("rows:", len(features))
print("columns:", len(features.columns))

for c in ["official_date", "game_datetime_utc"]:
    if c in features.columns:
        print(c, "min=", features[c].min(), "max=", features[c].max())

if TARGET_COL not in features.columns:
    raise ValueError(f"Missing target column: {TARGET_COL}")

completed = features[features[TARGET_COL].notna()].copy()
upcoming = features[features[TARGET_COL].isna()].copy()
completed[TARGET_COL] = completed[TARGET_COL].astype(int)

print("completed rows:", len(completed))
print("upcoming/unfinal rows:", len(upcoming))
display(completed[TARGET_COL].value_counts(dropna=False).to_frame("games"))
print("home win rate:", completed[TARGET_COL].mean())

## Baseline metrics

A model should beat the constant home-win-rate baseline on log loss and Brier score before you consider promoting it.

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, accuracy_score

home_rate = completed[TARGET_COL].mean()
y_all = completed[TARGET_COL].astype(int).to_numpy()
p_baseline = np.repeat(home_rate, len(y_all))

baseline_all = {
    "model_key": "baseline__constant_home_rate_all_completed",
    "n": len(y_all),
    "home_rate": home_rate,
    "log_loss": log_loss(y_all, p_baseline),
    "brier": brier_score_loss(y_all, p_baseline),
    "accuracy_if_always_home": accuracy_score(y_all, np.ones_like(y_all)),
    "roc_auc": 0.5,
}

pd.DataFrame([baseline_all])

## Feature discovery and exclusion rules

The model should use only pregame numeric features. Identifier columns, targets, final scores, labels, and raw market odds are excluded by default.

In [ ]:
def find_date_col(df: pd.DataFrame) -> str:
    for c in DATE_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No date column found from candidates: {DATE_COL_CANDIDATES}")

DATE_COL = find_date_col(completed)
print("DATE_COL:", DATE_COL)

# Convert date/time to datetime for chronological split.
completed[DATE_COL] = pd.to_datetime(completed[DATE_COL], errors="coerce", utc=True)
if len(upcoming):
    upcoming[DATE_COL] = pd.to_datetime(upcoming[DATE_COL], errors="coerce", utc=True)

EXCLUDE_EXACT = {
    TARGET_COL,
    "home_win", "away_win", "home_won", "away_won",
    "home_score", "away_score", "home_runs", "away_runs", "total_runs",
    "margin_home", "home_margin", "run_margin",
    "game_pk", "game_id", "request_id",
    "season", "game_year", "year",
}

# Conservative substring exclusions. Avoid excluding useful columns like runs_for_last10.
EXCLUDE_SUBSTRINGS = [
    "target_",
    "label_",
    "actual_",
    "post_",
    "final_score",
    "winner",
    "result",
]

MARKET_SUBSTRINGS = [
    "market_",
    "moneyline",
    "spread",
    "total_points",
    "over_price",
    "under_price",
    "book_count",
    "vig",
    "implied_prob",
]

META_SUBSTRINGS = [
    "name",
    "team_abbrev",
    "team_name",
    "pitcher_name",
    "venue",
    "status",
    "state",
    "datetime",
    "date",
]


def infer_numeric_feature_cols(df: pd.DataFrame, use_market: bool = False) -> list[str]:
    numeric_cols = df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    selected = []
    for c in numeric_cols:
        cl = c.lower()
        if c in EXCLUDE_EXACT:
            continue
        if any(s in cl for s in EXCLUDE_SUBSTRINGS):
            continue
        if any(s in cl for s in META_SUBSTRINGS):
            continue
        if (not use_market) and any(s in cl for s in MARKET_SUBSTRINGS):
            continue
        selected.append(c)
    return selected

feature_cols_raw = infer_numeric_feature_cols(completed, use_market=USE_MARKET_FEATURES)
print("raw numeric feature cols:", len(feature_cols_raw))
print(feature_cols_raw[:40])

## Missingness policy

Rules used here:

- Drop features with missingness greater than `DROP_IF_MISSING_GT`, default 35%.
- Keep features with 2%–35% missingness and let the model pipeline add missingness indicators through `SimpleImputer(add_indicator=True)`.
- Keep low-missingness columns and impute medians.

The stolen-base success-rate features are specifically prone to denominator missingness because teams may have zero steal attempts in a window.

In [ ]:
missing_pct = completed[feature_cols_raw].isna().mean().sort_values(ascending=False)
missing_table = missing_pct.to_frame("missing_pct")
display(missing_table.head(80))

# Explicit manual drops for unstable denominator-ratio features.
MANUAL_DROP_FEATURES = [
    "diff_team_box_sb_success_rate",
    "home_team_box_sb_success_rate",
    "away_team_box_sb_success_rate",
    "diff_team_box_sb_success_rate_last3",
]

high_missing_cols = missing_pct[missing_pct > DROP_IF_MISSING_GT].index.tolist()

drop_cols = sorted(set(high_missing_cols) | {c for c in MANUAL_DROP_FEATURES if c in feature_cols_raw})
feature_cols_clean = [c for c in feature_cols_raw if c not in drop_cols]

moderate_missing_cols = missing_pct[
    (missing_pct >= ADD_MISSING_FLAG_IF_MISSING_GT) &
    (missing_pct <= DROP_IF_MISSING_GT)
].index.tolist()
moderate_missing_cols = [c for c in moderate_missing_cols if c in feature_cols_clean]

print("Dropping columns:", len(drop_cols))
display(pd.DataFrame({"dropped_feature": drop_cols, "missing_pct": [missing_pct.get(c, np.nan) for c in drop_cols]}))

print("Clean feature cols:", len(feature_cols_clean))
print("Features with missing indicators added by imputer if missing in train:", len(moderate_missing_cols))
display(pd.DataFrame({"moderate_missing_feature": moderate_missing_cols[:100], "missing_pct": [missing_pct.get(c, np.nan) for c in moderate_missing_cols[:100]]}))

## Batted-ball velocity and distance features

Yes — you can calculate these features at the team level and compare them to the opponent.

Conceptually:

```text
team_avg_batted_ball_velocity = mean(launch_speed) for batted balls hit by that team
team_avg_batted_ball_distance = mean(hit_distance or hit_distance_sc) for batted balls hit by that team
matchup_diff_avg_ev = home_team_rolling_avg_ev - away_team_rolling_avg_ev
matchup_diff_avg_distance = home_team_rolling_avg_distance - away_team_rolling_avg_distance
```

In your current pipeline, average exit velocity is usually represented by columns containing `sc_avg_ev`. Distance may require an upstream aggregator addition if it is not already present in the Statcast aggregate tables.

In [ ]:
BB_VELOCITY_PATTERNS = [
    "avg_ev", "launch_speed", "exit_vel", "exit_velocity", "batted_ball_velocity"
]
BB_DISTANCE_PATTERNS = [
    "distance", "hit_distance", "batted_ball_distance"
]
BB_QUALITY_PATTERNS = BB_VELOCITY_PATTERNS + BB_DISTANCE_PATTERNS + [
    "barrel", "hard_hit", "sweetspot", "xwoba", "woba", "max_ev", "avg_la", "launch_angle"
]


def cols_matching(cols: list[str], patterns: list[str]) -> list[str]:
    patterns_l = [p.lower() for p in patterns]
    return [c for c in cols if any(p in c.lower() for p in patterns_l)]

velocity_cols = cols_matching(feature_cols_clean, BB_VELOCITY_PATTERNS)
distance_cols = cols_matching(feature_cols_clean, BB_DISTANCE_PATTERNS)
bb_quality_cols = cols_matching(feature_cols_clean, BB_QUALITY_PATTERNS)

print("Batted-ball velocity-like columns:", len(velocity_cols))
display(pd.DataFrame({"velocity_feature": velocity_cols[:100]}))

print("Batted-ball distance-like columns:", len(distance_cols))
display(pd.DataFrame({"distance_feature": distance_cols[:100]}))

print("All batted-ball/contact-quality columns:", len(bb_quality_cols))
display(pd.DataFrame({"bb_quality_feature": bb_quality_cols[:150]}))

In [ ]:
# Optional: inspect GCS-downloaded odds.db aggregate table schemas.
# This tells you whether average distance is available in your current SQLite aggregates.
if DB_PATH.exists():
    conn = sqlite3.connect(DB_PATH)
    for table in ["mlb_statcast_team_game", "mlb_statcast_team_hand_game", "mlb_statcast_pitcher_game"]:
        try:
            schema = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
            print("\n", table)
            display(schema[["name", "type"]])
        except Exception as e:
            print(table, "schema inspection failed:", e)
    conn.close()
else:
    print("No odds.db found at", DB_PATH)

### Upstream feature-engineering note for batted-ball distance

If the current aggregate tables do not include average batted-ball distance, add it upstream in the Statcast ingestion script. In Baseball Savant / pybaseball Statcast data, use batted-ball rows and aggregate `hit_distance` / `hit_distance_sc` when available.

Example aggregation pattern:

In [ ]:
# Example only. Put this pattern inside scripts/09_fetch_statcast.py where team-game Statcast aggregations are built.
# Column names can vary slightly by source/version; check whether your raw pybaseball frame has hit_distance_sc or hit_distance.

EXAMPLE_STATCAST_TEAM_DISTANCE_AGG = r'''
bbe = prepared.loc[prepared["launch_speed"].notna()].copy()

# Prefer hit_distance_sc if present; Baseball Savant CSV docs also refer to hit_distance.
distance_col = None
for candidate in ["hit_distance_sc", "hit_distance"]:
    if candidate in bbe.columns:
        distance_col = candidate
        break

agg_dict = {
    "sc_bbe": ("launch_speed", "count"),
    "sc_avg_ev": ("launch_speed", "mean"),
    "sc_max_ev": ("launch_speed", "max"),
    "sc_avg_la": ("launch_angle", "mean"),
}

if distance_col:
    agg_dict["sc_avg_batted_ball_distance"] = (distance_col, "mean")
    agg_dict["sc_max_batted_ball_distance"] = (distance_col, "max")

team_game = (
    bbe.groupby(["game_pk", "official_date", "batting_team_id"], as_index=False)
       .agg(**agg_dict)
)
'''

print(EXAMPLE_STATCAST_TEAM_DISTANCE_AGG)

## Chronological train/test split

In [ ]:
model_df = completed.sort_values([DATE_COL, "game_pk" if "game_pk" in completed.columns else DATE_COL]).reset_index(drop=True)

split_idx = int(len(model_df) * (1 - TEST_SIZE))
train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

print("train rows:", len(train_df), train_df[DATE_COL].min(), train_df[DATE_COL].max())
print("test rows:", len(test_df), test_df[DATE_COL].min(), test_df[DATE_COL].max())
print("train home rate:", train_df[TARGET_COL].mean())
print("test home rate:", test_df[TARGET_COL].mean())

# Holdout baseline uses train home rate, not full-sample home rate.
y_test = test_df[TARGET_COL].astype(int).to_numpy()
p_test_base = np.repeat(train_df[TARGET_COL].mean(), len(y_test))

holdout_baseline = {
    "feature_set": "baseline",
    "model_name": "train_home_rate_constant",
    "n": len(y_test),
    "log_loss": log_loss(y_test, p_test_base),
    "brier": brier_score_loss(y_test, p_test_base),
    "roc_auc": 0.5,
    "accuracy_50pct": accuracy_score(y_test, (p_test_base >= 0.5).astype(int)),
    "avg_pred": p_test_base.mean(),
    "actual_rate": y_test.mean(),
}

pd.DataFrame([holdout_baseline])

## Define feature sets

These feature sets isolate whether the signal comes from Elo, starter Statcast, team contact quality, bullpen, or the full enhanced feature space.

In [ ]:
def has_any(c: str, patterns: list[str]) -> bool:
    cl = c.lower()
    return any(p.lower() in cl for p in patterns)


def pick(patterns: list[str], cols: list[str] = None) -> list[str]:
    cols = feature_cols_clean if cols is None else cols
    return [c for c in cols if has_any(c, patterns)]

FEATURE_SETS = {
    "elo_only": pick(["elo"]),
    "baseline_team_form": pick([
        "win_last", "win_season", "run_diff", "runs_for", "runs_against",
        "home_field", "rest", "days_rest", "travel"
    ]),
    "starter_boxscore_only": [
        c for c in feature_cols_clean
        if "starter" in c.lower() and "statcast" not in c.lower() and "sc_" not in c.lower()
    ],
    "starter_statcast_only": pick(["starter_statcast", "starter_sc_", "starter_statcast_sc_"]),
    "starter_all": [c for c in feature_cols_clean if "starter" in c.lower()],
    "bullpen_all": pick(["bullpen"]),
    "statcast_team_only": pick(["team_off_sc", "team_vs_hand_sc", "sc_avg_ev", "sc_xwoba", "sc_woba", "sc_barrel", "sc_hard_hit", "sc_sweetspot"]),
    "batted_ball_quality": bb_quality_cols,
    "enhanced_no_market": feature_cols_clean,
}

# Remove empty sets and dedupe columns while preserving order.
def dedupe(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

FEATURE_SETS = {k: dedupe(v) for k, v in FEATURE_SETS.items() if len(v) > 0}

summary = []
for name, cols in FEATURE_SETS.items():
    summary.append({
        "feature_set": name,
        "feature_count": len(cols),
        "missing_mean": float(completed[cols].isna().mean().mean()) if cols else np.nan,
        "missing_max": float(completed[cols].isna().mean().max()) if cols else np.nan,
    })

feature_set_summary = pd.DataFrame(summary).sort_values("feature_count")
display(feature_set_summary)

## Model definitions

All pipelines include median imputation with missing indicators. This means the exported estimator can receive the original feature columns directly; it does not require you to manually create `__is_missing` columns in production.

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform


def build_model(model_name: str):
    if model_name == "logit_l2":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, penalty="l2", solver="lbfgs", random_state=RANDOM_STATE)),
        ])
    if model_name == "random_forest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight=None)),
        ])
    if model_name == "extra_trees":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight=None)),
        ])
    if model_name == "hist_gradient_boosting":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE, l2_regularization=0.0)),
        ])
    if model_name == "xgboost":
        from xgboost import XGBClassifier
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])
    if model_name == "lightgbm":
        from lightgbm import LGBMClassifier
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", LGBMClassifier(
                objective="binary",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            )),
        ])
    if model_name == "svm_rbf":
        from sklearn.svm import SVC
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True, random_state=RANDOM_STATE)),
        ])
    raise ValueError(model_name)


def param_distributions(model_name: str):
    if model_name == "logit_l2":
        return {"model__C": loguniform(0.01, 10.0)}
    if model_name == "random_forest":
        return {
            "model__n_estimators": randint(300, 900),
            "model__max_depth": [3, 4, 5, 6, 8, None],
            "model__min_samples_leaf": randint(10, 80),
            "model__max_features": ["sqrt", "log2", 0.35, 0.50, 0.75],
        }
    if model_name == "extra_trees":
        return {
            "model__n_estimators": randint(300, 900),
            "model__max_depth": [3, 4, 5, 6, 8, None],
            "model__min_samples_leaf": randint(10, 80),
            "model__max_features": ["sqrt", "log2", 0.35, 0.50, 0.75],
        }
    if model_name == "hist_gradient_boosting":
        return {
            "model__learning_rate": loguniform(0.01, 0.20),
            "model__max_iter": randint(100, 500),
            "model__max_leaf_nodes": randint(8, 40),
            "model__min_samples_leaf": randint(15, 80),
            "model__l2_regularization": loguniform(0.001, 10.0),
        }
    if model_name == "xgboost":
        return {
            "model__n_estimators": randint(150, 700),
            "model__max_depth": randint(2, 6),
            "model__learning_rate": loguniform(0.01, 0.12),
            "model__subsample": uniform(0.65, 0.30),
            "model__colsample_bytree": uniform(0.65, 0.30),
            "model__min_child_weight": randint(5, 30),
            "model__reg_alpha": loguniform(0.001, 2.0),
            "model__reg_lambda": loguniform(0.5, 8.0),
        }
    if model_name == "lightgbm":
        return {
            "model__n_estimators": randint(150, 700),
            "model__learning_rate": loguniform(0.01, 0.12),
            "model__num_leaves": randint(8, 48),
            "model__min_child_samples": randint(15, 100),
            "model__subsample": uniform(0.65, 0.30),
            "model__colsample_bytree": uniform(0.65, 0.30),
            "model__reg_alpha": loguniform(0.001, 2.0),
            "model__reg_lambda": loguniform(0.5, 8.0),
        }
    if model_name == "svm_rbf":
        return {
            "model__C": loguniform(0.05, 10.0),
            "model__gamma": loguniform(0.0001, 0.1),
        }
    return None

MODEL_NAMES = ["logit_l2", "random_forest", "extra_trees", "hist_gradient_boosting"]
if RUN_XGBOOST:
    MODEL_NAMES.append("xgboost")
if RUN_LIGHTGBM:
    MODEL_NAMES.append("lightgbm")
if RUN_SVM:
    MODEL_NAMES.append("svm_rbf")

MODEL_NAMES

## Training helpers

In [ ]:
def make_X(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    # Keep original columns only. Missing indicators are added inside SimpleImputer(add_indicator=True).
    return df[cols].copy()


def clipped_probs(p):
    return np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)


def evaluate_probs(y_true, p, threshold: float = 0.5) -> dict:
    p = clipped_probs(p)
    y_true = np.asarray(y_true).astype(int)
    out = {
        "n": len(y_true),
        "log_loss": log_loss(y_true, p),
        "brier": brier_score_loss(y_true, p),
        "accuracy_50pct": accuracy_score(y_true, (p >= threshold).astype(int)),
        "avg_pred": float(np.mean(p)),
        "actual_rate": float(np.mean(y_true)),
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, p)
    except Exception:
        out["roc_auc"] = np.nan
    return out


def fit_one_model(model_name: str, X_train, y_train):
    pipe = build_model(model_name)
    params = param_distributions(model_name)

    if USE_RANDOMIZED_SEARCH and params:
        cv = TimeSeriesSplit(n_splits=CV_SPLITS)
        search = RandomizedSearchCV(
            pipe,
            param_distributions=params,
            n_iter=N_ITER,
            scoring="neg_log_loss",
            cv=cv,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=0,
            refit=True,
        )
        search.fit(X_train, y_train)
        return search.best_estimator_, search.best_params_, -search.best_score_
    else:
        pipe.fit(X_train, y_train)
        return pipe, {}, np.nan

## Run model comparison

This cell can take several minutes depending on feature count and model list.

In [ ]:
y_train = train_df[TARGET_COL].astype(int).to_numpy()
y_test = test_df[TARGET_COL].astype(int).to_numpy()

results = []
fitted = {}

# Add baseline row first.
results.append(holdout_baseline)

for feature_set_name, cols in FEATURE_SETS.items():
    if len(cols) == 0:
        continue

    X_train = make_X(train_df, cols)
    X_test = make_X(test_df, cols)

    for model_name in MODEL_NAMES:
        print(f"Fitting {feature_set_name}__{model_name} with {len(cols)} features...")
        try:
            estimator, best_params, cv_log_loss = fit_one_model(model_name, X_train, y_train)
            p_test = estimator.predict_proba(X_test)[:, 1]
            metrics = evaluate_probs(y_test, p_test)
            row = {
                "feature_set": feature_set_name,
                "model_name": model_name,
                "feature_count": len(cols),
                "cv_log_loss": cv_log_loss,
                "best_params": best_params,
                **metrics,
            }
            results.append(row)
            fitted[(feature_set_name, model_name)] = {
                "estimator": estimator,
                "feature_cols": cols,
                "test_probs": p_test,
                "best_params": best_params,
                "metrics": metrics,
            }
        except Exception as e:
            print(f"FAILED {feature_set_name}__{model_name}: {type(e).__name__}: {e}")
            results.append({
                "feature_set": feature_set_name,
                "model_name": model_name,
                "feature_count": len(cols),
                "error": f"{type(e).__name__}: {e}",
            })

results_df = pd.DataFrame(results)
metric_cols = ["feature_set", "model_name", "feature_count", "n", "log_loss", "brier", "roc_auc", "accuracy_50pct", "avg_pred", "actual_rate", "cv_log_loss"]
display(results_df[[c for c in metric_cols if c in results_df.columns]].sort_values(["log_loss", "brier"], na_position="last").head(30))

## Interpret model comparison

Look for a model that beats the baseline row on log loss and Brier score. If a model only improves accuracy but worsens log loss or calibration, do not promote it.

In [ ]:
ranked = results_df.dropna(subset=["log_loss", "brier"]).sort_values(["log_loss", "brier"]).reset_index(drop=True)
display(ranked.head(20))

best_row = ranked.iloc[0].to_dict()
print("Best by log_loss:")
print(json.dumps({k: str(v) for k, v in best_row.items() if k != "best_params"}, indent=2))
print("Best params:", best_row.get("best_params"))

## Calibration table and chart

In [ ]:
def calibration_table(y_true, p, bins=None):
    if bins is None:
        bins = [0.0, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 1.0]
    df = pd.DataFrame({"y": y_true, "p": clipped_probs(p)})
    df["prob_bucket"] = pd.cut(df["p"], bins=bins, include_lowest=True)
    tab = df.groupby("prob_bucket", observed=True).agg(
        games=("y", "size"),
        avg_pred_prob=("p", "mean"),
        actual_home_win_rate=("y", "mean"),
    )
    tab["calibration_error"] = tab["actual_home_win_rate"] - tab["avg_pred_prob"]
    return tab

best_key = (best_row["feature_set"], best_row["model_name"])
best_obj = fitted.get(best_key)
if best_obj is None:
    raise ValueError(f"Best model object not found for {best_key}")

best_probs = best_obj["test_probs"]
calib = calibration_table(y_test, best_probs)
display(calib)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
plot_tab = calib.dropna().copy()
ax.plot(plot_tab["avg_pred_prob"], plot_tab["actual_home_win_rate"], marker="o")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_xlabel("Average predicted home-win probability")
ax.set_ylabel("Actual home-win rate")
ax.set_title(f"Calibration: {best_key[0]}__{best_key[1]}")
ax.grid(True, alpha=0.3)
plt.show()

## Classification report and confusion matrix at 50% threshold

This is useful diagnostic information, but it is not the primary promotion criterion for betting.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_50 = (best_probs >= 0.5).astype(int)
print(classification_report(y_test, pred_50, target_names=["actual_away", "actual_home"], digits=3))
cm = pd.DataFrame(
    confusion_matrix(y_test, pred_50),
    index=["actual_away", "actual_home"],
    columns=["pred_away", "pred_home"],
)
display(cm)

## Permutation feature importance for champion candidate

This can be slow for very large feature sets. Use `N_REPEATS=5` for a quick view and increase later.

In [ ]:
from sklearn.inspection import permutation_importance

RUN_PERMUTATION_IMPORTANCE = True
N_REPEATS = 5
MAX_IMPORTANCE_ROWS = 50

if RUN_PERMUTATION_IMPORTANCE:
    X_test_best = make_X(test_df, best_obj["feature_cols"])
    imp = permutation_importance(
        best_obj["estimator"],
        X_test_best,
        y_test,
        scoring="neg_log_loss",
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    importance_df = pd.DataFrame({
        "feature": best_obj["feature_cols"],
        "importance_mean": imp.importances_mean,
        "importance_std": imp.importances_std,
    }).sort_values("importance_mean", ascending=False)
    display(importance_df.head(MAX_IMPORTANCE_ROWS))
else:
    print("Permutation importance skipped.")

## Optional market/edge analysis

This section only works if the feature file contains historical market fields. It does not train on market columns by default, but it can compare model probability to market no-vig probability when available.

In [ ]:
market_candidates = [
    "market_home_no_vig_prob",
    "market_away_no_vig_prob",
    "home_moneyline_median",
    "away_moneyline_median",
]
existing_market_cols = [c for c in market_candidates if c in test_df.columns]
print("Existing market columns:", existing_market_cols)

if "market_home_no_vig_prob" in test_df.columns:
    edge_df = test_df[["game_pk" if "game_pk" in test_df.columns else DATE_COL, TARGET_COL, "market_home_no_vig_prob"]].copy()
    edge_df["model_home_win_prob"] = best_probs
    edge_df["model_edge"] = edge_df["model_home_win_prob"] - edge_df["market_home_no_vig_prob"]
    display(edge_df[["model_edge", "model_home_win_prob", "market_home_no_vig_prob", TARGET_COL]].describe())

    edge_bins = [-1, -0.05, -0.02, 0, 0.02, 0.05, 1]
    edge_df["edge_bucket"] = pd.cut(edge_df["model_edge"], bins=edge_bins)
    edge_summary = edge_df.groupby("edge_bucket", observed=True).agg(
        games=(TARGET_COL, "size"),
        avg_edge=("model_edge", "mean"),
        avg_model_prob=("model_home_win_prob", "mean"),
        avg_market_prob=("market_home_no_vig_prob", "mean"),
        actual_home_win_rate=(TARGET_COL, "mean"),
    )
    display(edge_summary)
else:
    print("market_home_no_vig_prob not present. Edge analysis skipped for now.")

## Champion export

This exports a bundle containing the fitted estimator, feature columns, metrics, and missingness policy. The estimator itself has median imputation with missing indicators inside its pipeline.

Leave `APPROVE_EXPORT = False` until you review the results.

In [ ]:
import joblib

if CHAMPION_KEY is None:
    champion_key = best_key
else:
    champion_key = CHAMPION_KEY

champion_obj = fitted.get(champion_key)
if champion_obj is None:
    raise ValueError(f"Champion key not found: {champion_key}")

champion_feature_set, champion_model_name = champion_key
champion_metrics = champion_obj["metrics"]
champion_cols = champion_obj["feature_cols"]

bundle = {
    "estimator": champion_obj["estimator"],
    "feature_cols": champion_cols,
    "feature_set": champion_feature_set,
    "model_name": champion_model_name,
    "target_col": TARGET_COL,
    "date_col": DATE_COL,
    "missing_policy": {
        "drop_if_missing_gt": DROP_IF_MISSING_GT,
        "add_missing_flag_if_missing_gt": ADD_MISSING_FLAG_IF_MISSING_GT,
        "dropped_cols": drop_cols,
        "imputer_add_indicator": True,
    },
    "metrics": champion_metrics,
    "best_params": champion_obj.get("best_params", {}),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "features_path": str(FEATURES_PATH),
}

metadata = {k: v for k, v in bundle.items() if k != "estimator"}
metadata["feature_count"] = len(champion_cols)
metadata["feature_cols"] = champion_cols

print("Candidate champion:", champion_key)
print("Feature count:", len(champion_cols))
print("Metrics:")
print(json.dumps(champion_metrics, indent=2))

if APPROVE_EXPORT:
    model_path = MODEL_DIR / f"{CHAMPION_EXPORT_NAME}.joblib"
    metadata_path = MODEL_DIR / f"{CHAMPION_EXPORT_NAME}_metadata.json"

    joblib.dump(bundle, model_path)
    metadata_path.write_text(json.dumps(metadata, indent=2, default=str))

    print("Exported:", model_path)
    print("Exported:", metadata_path)
else:
    print("APPROVE_EXPORT is False. Review results before exporting.")

## Production scoring reminder

Because this notebook exports a bundle with `feature_cols`, the production scorer should:

1. Load the bundle using `joblib.load`.
2. Select `features[bundle["feature_cols"]]`.
3. Pass that frame directly into `bundle["estimator"].predict_proba(...)`.

The estimator pipeline handles median imputation and missing indicators internally.